クラウド学習トライアル

こちらのプロジェクトはhello-worldプロジェクトで作成中の指の数を数えるロジックをMedia PipeからYOLOに載せ替える際に必要となる学習データの作成のためのスクリプトです。

内容は次の通り
アップロードされた学習用の動画をCOCOデータセットで解析。解析データをアノテーションにして学習を行わせます。鶏が先か、卵が先か。。。

1.   動作準備。GoogleDriveのマウントと必要なパッケージのインストール
2.   動画からフレームを取り出して学習データに変換
3.   機械学習





In [1]:
# 1. Google DriveをColabにマウント（接続）します
from google.colab import drive

drive.mount("/content/drive")

# 2. YOLO（Ultralytics）とMediaPipeをColab環境にインストールします
!pip install ultralytics mediapipe opencv-python

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [14]:
import os
import cv2
from ultralytics import YOLO

# =====================================================================
# パス設定
# =====================================================================
VIDEO_PATH = "/content/drive/MyDrive/AI_study/my_hand_video.mp4"
DATASET_DIR = "/content/drive/MyDrive/AI_study/yolo_dataset"

SAVE_INTERVAL_FRAMES = 10  # 10フレームごとに1枚抽出

# フォルダ自動作成
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
LABELS_DIR = os.path.join(DATASET_DIR, "labels")
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(LABELS_DIR, exist_ok=True)

# =====================================================================
# 💡 解決策：MediaPipeを捨てて、YOLO純正の自動データ抽出に切り替えます
# =====================================================================
print("=== YOLO純正システムによるデータセット抽出スタート ===")

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print(f"エラー: 動画が見つかりません: {VIDEO_PATH}")
else:
    # 1. 裏側でアノテーションの先生役となる、一番賢いYOLOの巨大モデルを読み込みます
    # (自動でダウンロードされます。MediaPipeのようなパスバグは一切起きません)
    teacher_model = YOLO("yolov8x.pt")

    frame_count = 0
    saved_count = 0

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame_count += 1
        if frame_count % SAVE_INTERVAL_FRAMES != 0:
            continue

        # YOLOの先生モデルに画像を読ませて、手（またはオブジェクト）の位置を判定させます
        # 画面に描画せず、裏の計算（推論）だけを爆速で行います
        results = teacher_model.predict(frame, verbose=False)

        # 先生モデルが何かを検出した場合
        if len(results[0].boxes) > 0:
            saved_count += 1
            file_base = f"frame_{frame_count:05d}"

            # --- 1. 通常データの保存 ---
            img_path_norm = os.path.join(IMAGES_DIR, f"{file_base}.jpg")
            txt_path_norm = os.path.join(LABELS_DIR, f"{file_base}.txt")

            # 画像の保存
            cv2.imwrite(img_path_norm, frame, [int(cv2.IMWRITE_JPEG_QUALITY), 95])

            # YOLOの予測結果（テキスト）をそのままlabelsフォルダに書き出します
            results[0].save_txt(txt_path_norm)

            # --- 2. 左右反転データの保存（データかさ増し） ---
            flipped_frame = cv2.flip(frame, 1)
            img_path_flip = os.path.join(IMAGES_DIR, f"{file_base}_flip.jpg")
            txt_path_flip = os.path.join(LABELS_DIR, f"{file_base}_flip.txt")

            cv2.imwrite(
                img_path_flip,
                flipped_frame,
                [int(cv2.IMWRITE_JPEG_QUALITY), 95],
            )

            # 反転させた画像からもYOLOにアノテーションを自動計算させます
            flipped_results = teacher_model.predict(flipped_frame, verbose=False)
            flipped_results[0].save_txt(txt_path_flip)

            print(f"進行度: {saved_count * 2} ファイル生成中...")

    cap.release()
    print(
        f"=== 【成功】生成完了: 合計 {saved_count * 2} ファイルがDriveに保存されました！ ==="
    )


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
=== YOLO純正システムによるデータセット抽出スタート ===
進行度: 2 ファイル生成中...
進行度: 4 ファイル生成中...
進行度: 6 ファイル生成中...
進行度: 8 ファイル生成中...
進行度: 10 ファイル生成中...
進行度: 12 ファイル生成中...
進行度: 14 ファイル生成中...
進行度: 16 ファイル生成中...
進行度: 18 ファイル生成中...
進行度: 20 ファイル生成中...
進行度: 22 ファイル生成中...
進行度: 24 ファイル生成中...
進行度: 26 ファイル生成中...
進行度: 28 ファイル生成中...
進行度: 30 ファイル生成中...
進行度: 32 ファイル生成中...
進行度: 34 ファイル生成中...
進行度: 36 ファイル生成中...
進行度: 38 ファイル生成中...
進行度: 40 ファイル生成中...
進行度: 42 ファイル生成中...
進行度: 44 ファイル生成中...
進行度: 46 ファイル生成中...
進行度: 48 ファイル生成中...
進行度: 50 ファイル生成中...
進行度: 52 ファイル生成中...
進行度: 54 ファイル生成中...
進行度: 56 ファイル生成中...
進行度: 58 ファイル生成中...
進行度: 60 ファイル生成中...
進行度: 62 ファイル生成中...
進行度: 64 ファイル生成中...
進行度: 66 ファイル生成中...
進行度: 68 ファイル生成中...
進行度: 70 ファイル生成

In [2]:
# 💡 再掲：【セル3】の学習スタート用コードです
from ultralytics import YOLO
import shutil
import os

# 1. YOLOにデータセットの場所を教える設定ファイル(dataset.yaml)を自動生成
# (前回のセル2で yolo_dataset フォルダがDrive内に作られている前提です)
yaml_content = """
path: /content/drive/MyDrive/AI_study/yolo_dataset
train: images
val: images

names:
  0: zero
  1: one
  2: two
  3: three
  4: four
  5: five
"""

with open("/content/dataset.yaml", "w") as f:
    f.write(yaml_content.strip())

# 2. クラウドGPU(device=0)を使って、生徒モデル(Nano)を100エポック学習させます！
model = YOLO("yolov8n.pt")
model.train(data="/content/dataset.yaml", epochs=100, imgsz=640, device=0)

# 3. 完成した最高精度の重みファイル（best.pt）を、Google Driveのわかりやすい場所にコピー
os.makedirs("/content/drive/MyDrive/AI_study/output", exist_ok=True)
shutil.copy("/content/runs/detect/train/weights/best.pt", "/content/drive/MyDrive/AI_study/output/best.pt")
print("🎉 学習完了！ Google Driveの 'AI_study/output/best.pt' にモデルが保存されました！")



Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=t